# Mar 10/11 · 09:45 — TP v0.6 : venv, dépendances et variables d'environnement

Mar 10/11, TP de 10:35 à 11:10, après le CC1. Correction à 11:10, TP fini ou non. La référence v0.6 est publiée à 11:15.

## Objectif

`fleetcheck` se prépare à parler à des services web (v0.9) et à alerter un salon d'astreinte (v0.10) :

- vous savez quel Python exécute votre code et où s'installent ses paquets ;
- `requests` est installé et la dépendance est figée dans `requirements.txt` ;
- l'URL du webhook d'alerte se lit dans l'environnement, jamais dans le code, le dépôt ou le journal.

## Mode d'emploi du notebook

- Copiez ce notebook à la racine de votre dépôt, à côté de `fleetcheck.py`. Le code s'écrit dans `fleetcheck.py`, ouvert dans un onglet de JupyterLab ; le notebook sert à essayer et à vérifier chaque étape.
- Exécutez d'abord la cellule d'outils ci-dessous. Chaque vérification affiche `OK`, ou `KO` avec ce qui était attendu.
- `%run -n fleetcheck.py` charge les fonctions du fichier enregistré sans exécuter le bloc `if __name__ == "__main__":`. N'appelez pas `main()` dans le notebook : argparse y lirait les arguments du noyau Jupyter.
- `run(...)` lance le script comme dans un terminal, avec ses arguments, et renvoie `r.stdout`, `r.stderr` et `r.returncode`. Le script lancé hérite des variables d'environnement du noyau : c'est ce que l'étape 3 utilise.
- Les vérifications qui modifient l'environnement ou un fichier remettent tout en place : ne les modifiez pas.
- **Prédire** et **Justifier** marquent les moments clés du TP. Répondez en une ou deux lignes dans la cellule `# Ma réponse :` qui suit, **avant** de lire la suite ou d'exécuter quoi que ce soit. Les réponses sont reprises à la correction.
- Rendu en fin de TP : menu *Kernel > Restart Kernel and Run All Cells*, puis commit et push de `fleetcheck.py`, `config.json`, `requirements.txt`, `servers.csv`, `servers-broken.csv` et de ce notebook.

In [ ]:
# Outils du TP : exécuter cette cellule en premier, sans la modifier.
import os
import subprocess
import sys


def check(obtenu, attendu):
    if obtenu == attendu:
        print("OK")
    elif isinstance(attendu, str) and "\n" in attendu:
        print(f"KO, obtenu :\n{obtenu}\nattendu :\n{attendu}")
    else:
        print(f"KO\n  obtenu  : {obtenu!r}\n  attendu : {attendu!r}")


def check_error(fonction, *args, attendu):
    try:
        fonction(*args)
    except Exception as e:
        check(f"{type(e).__name__}: {e}", attendu)
    else:
        print(f"KO, aucune exception levée. Attendu : {attendu}")


def run(*args):
    """Lance fleetcheck.py comme dans un terminal (vu en v0.8 : subprocess)."""
    r = subprocess.run([sys.executable, "fleetcheck.py", *args], capture_output=True,
                       encoding="utf-8", env={**os.environ, "PYTHONUTF8": "1", "NO_COLOR": "1",
                                              "PYTHONWARNDEFAULTENCODING": "1"})
    print(r.stdout, end="")
    print(r.stderr, end="", file=sys.stderr)
    print(f"[code de sortie : {r.returncode}]")
    if "EncodingWarning" in r.stderr:
        print("KO : un open(), read_text() ou subprocess text=True sans encoding= (ligne indiquée dans l'EncodingWarning ci-dessus)")
    return r

## Point de départ

Votre `fleetcheck.py` de v0.5, avec `config.json`, `servers.csv` et `servers-broken.csv` dans le même dossier.

En retard ou en panne : copiez dans votre dépôt les fichiers de `reference/` au tag `v0.5` du dépôt public (`git switch --detach v0.5` dans votre clone de consultation).

Vérifiez avant de commencer :

In [ ]:
RESUME = """9 serveurs valides, 0 invalides
par env : prod 7, staging 2
par rôle : web 3, api 3, worker 1, cache 1, db 1
prod/web : web-1, web-2
"""

os.environ.pop("FLEETCHECK_WEBHOOK_URL", None)
r = run()
check((r.stdout, r.stderr, r.returncode), (RESUME, "", 0))

## Étapes

| Fin | Étape |
|---|---|
| 10:45 | 1. Vérifier que le noyau tourne dans le venv, hors de votre dépôt |
| 10:55 | 2. Installer `requests` et écrire `requirements.txt` |
| 11:10 | 3. Lire `FLEETCHECK_WEBHOOK_URL` dans l'environnement |

À 10:55, si plus d'un tiers de la salle n'a pas `requests` importable, l'enseignant installe au vidéoprojecteur et la fin de l'étape 1 passe en devoir.

### 1. Quel Python ? — fin 10:45

Le 12/10, vous avez créé `esgi-python/.venv` et lancé JupyterLab avec son `python`. Le noyau de ce notebook est donc ce Python-là.

- `sys.executable` : le chemin de l'interpréteur qui exécute le noyau.
- `sys.prefix` : le dossier de l'environnement (`…/esgi-python/.venv`) ; `sys.base_prefix` : le Python d'origine, installé depuis python.org.

In [ ]:
import sys

print(sys.executable)
print(sys.prefix)
print(sys.base_prefix)

**Prédire** — Que vaut `sys.prefix == sys.base_prefix` dans ce notebook ?

In [ ]:
# Ma réponse :

Vérification : le noyau tourne dans un venv, et ce venv est hors de votre dépôt (`esgi-python/.venv`, à côté de `fleetcheck-<nom>/`) : git ne le voit pas. Un venv créé dans un dépôt se met dans le `.gitignore` : il se recrée, il ne se commite pas.

In [ ]:
from pathlib import Path

check(sys.prefix != sys.base_prefix, True)
check(Path(sys.prefix).resolve().is_relative_to(Path.cwd().resolve()), False)

### 2. `requests` et `requirements.txt` — fin 10:55

`%pip` installe dans le Python du noyau, celui de l'étape 1. Sans réseau en salle, l'enseignant donne la commande de repli (`--no-index --find-links wheels`).

In [ ]:
%pip install requests==2.34.2

Écrivez ensuite `requirements.txt` à la racine de votre dépôt, une ligne par dépendance directe, version épinglée par `==` :

```text
requests==2.34.2
```

**Justifier** — Pourquoi `requests==2.34.2` plutôt que `requests` seul ?

In [ ]:
# Ma réponse :

Vérification : `requests` s'importe dans la bonne version, et chaque ligne non vide de `requirements.txt` a la forme `nom==version`.

In [ ]:
import re
import requests

check(requests.__version__, "2.34.2")
lignes = [l.strip() for l in Path("requirements.txt").read_text(encoding="utf-8").splitlines() if l.strip()]
check(all(re.fullmatch(r"[A-Za-z0-9._-]+==[0-9][A-Za-z0-9.]*", l) for l in lignes), True)
check("requests==2.34.2" in lignes, True)

Un collègue ou la CI reconstruit l'environnement par `python -m pip install -r requirements.txt`. `pip freeze` liste tout ce qui est installé, dépendances de `requests` comprises (`urllib3`, `idna`…) : n'y gardez que ce que le projet importe.

### 3. `FLEETCHECK_WEBHOOK_URL` — fin 11:10

En v0.10, `fleetcheck` postera une alerte vers un webhook (Discord, Slack…). Son URL est un secret : quiconque la connaît poste dans le salon.

Dans `main()`, après le chargement de l'inventaire et de la config :

- lisez la variable par `os.environ.get("FLEETCHECK_WEBHOOK_URL")` ;
- présente : log `INFO FLEETCHECK_WEBHOOK_URL défini : alerte activée` ;
- absente : log `INFO FLEETCHECK_WEBHOOK_URL absent : pas d'alerte` ;
- jamais la valeur, ni dans le journal ni sur stdout : le log de CI est lisible par toute l'équipe.

Pour essayer dans un terminal : `export FLEETCHECK_WEBHOOK_URL=http://127.0.0.1:8000/webhook` (bash, zsh), `$env:FLEETCHECK_WEBHOOK_URL = "http://127.0.0.1:8000/webhook"` (PowerShell). Dans le notebook, `os.environ[...] = ...` suffit : `run(...)` lance un processus enfant, qui hérite de l'environnement du noyau.

**Justifier** — Pourquoi l'URL du webhook dans l'environnement plutôt que dans `config.json` ?

In [ ]:
# Ma réponse :

Vérification, variable absente :

In [ ]:
os.environ.pop("FLEETCHECK_WEBHOOK_URL", None)
r = run("-v")
check("INFO FLEETCHECK_WEBHOOK_URL absent : pas d'alerte" in r.stderr.splitlines(), True)
check((r.stdout, r.returncode), (RESUME, 0))

Vérification, variable posée, puis retirée : l'URL n'apparaît nulle part.

In [ ]:
URL = "http://127.0.0.1:8000/webhook"
os.environ["FLEETCHECK_WEBHOOK_URL"] = URL
try:
    r = run("-v")
    sans_option = run()
finally:
    del os.environ["FLEETCHECK_WEBHOOK_URL"]
check("INFO FLEETCHECK_WEBHOOK_URL défini : alerte activée" in r.stderr.splitlines(), True)
check(URL in r.stderr or URL in r.stdout, False)
check((sans_option.stderr, sans_option.returncode), ("", 0))

Log attendu avec `-v`, variable absente (référence v0.6) :

```text
INFO 9 serveur(s) valide(s), 0 ligne(s) invalide(s) dans servers.csv
INFO FLEETCHECK_WEBHOOK_URL absent : pas d'alerte
INFO rapport écrit : report.json
```

## Critère final

stdout, stderr et code de sortie ne changent pas depuis v0.5 quand la variable est absente :

| Lancement | stdout | stderr | Code |
|---|---|---|---|
| `run()` | le résumé | rien | `0` |
| `run("--inventory", "servers-broken.csv")` | le résumé | les 3 `WARNING` | `2` |
| `run("--env", "prd")` | rien | `ERROR aucun serveur pour --env prd` | `2` |

In [ ]:
os.environ.pop("FLEETCHECK_WEBHOOK_URL", None)
r = run()
check((r.stdout, r.stderr, r.returncode), (RESUME, "", 0))
r = run("--inventory", "servers-broken.csv")
check((r.stdout.splitlines()[0], r.stderr.count("WARNING ligne"), r.returncode), ("9 serveurs valides, 3 invalides", 3, 2))
r = run("--env", "prd")
check((r.stdout, r.stderr, r.returncode), ("", "ERROR aucun serveur pour --env prd\n", 2))

## Pour aller plus loin

- Créez un venv neuf dans un dossier temporaire (`python3.14 -m venv /tmp/essai`, sous Windows `py -3.14 -m venv $env:TEMP\essai`), puis `…/bin/python -m pip install -r requirements.txt` : `requests` s'importe dans ce venv, sans rien d'autre que ce que liste le fichier.
- `os.environ.get("FLEETCHECK_ATTEMPTS", "3")` renvoie une chaîne : faites-en un entier, et un `ERROR` en code `2` si la valeur n'est pas un nombre.